# 04 — Start AIC Demo Server on Kaggle

Notebook này không build lại visual/OCR. Nó đọc trực tiếp output của `00-note`, `01-note`, `02-note`, khởi động FastAPI và chạy smoke benchmark.

Attach trước:

- `nguyenminhtric/00-note`
- `nguyenminhtric/01-note`
- `yixuanisthebest/02-note`
- dataset `nguynnc/aic2025`

Đưa repo `aic_latency_demo` vào `/kaggle/working` bằng Git clone hoặc upload ZIP rồi giải nén.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path("/kaggle/working/aic_latency_demo")
assert PROJECT_ROOT.is_dir(), f"Missing repo: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# Minimal dependencies. FAISS is optional because the server has a NumPy fallback.
!python -m pip install -q -r requirements.txt transformers

In [ ]:
import os

os.environ["AIC_MODE"] = "artifact"
os.environ["AIC_VISUAL_MANIFEST"] = "/kaggle/input/notebooks/nguyenminhtric/01-note/aic2025_full_build/artifacts/visual_index/visual_index_manifest.json"
os.environ["AIC_OCR_SQLITE"] = "/kaggle/input/notebooks/yixuanisthebest/02-note/aic2025_full_build/artifacts/ocr_index/ocr_index.sqlite"
os.environ["AIC_MODEL_DIR"] = "/kaggle/input/notebooks/nguyenminhtric/00-note/aic2025_full_build/artifacts/weights/siglip2_so400m_patch14_384"
os.environ["AIC_DATASET_ROOT"] = "/kaggle/input/datasets/nguynnc/aic2025"
os.environ["AIC_PROJECT_ROOTS"] = ",".join([
    "/kaggle/input/notebooks/nguyenminhtric/00-note/aic2025_full_build",
    "/kaggle/input/notebooks/nguyenminhtric/01-note/aic2025_full_build",
    "/kaggle/input/notebooks/yixuanisthebest/02-note/aic2025_full_build",
])
os.environ["AIC_CORS_ORIGINS"] = "*"
os.environ["AIC_PORT"] = "8000"
print("Artifact environment configured")

In [ ]:
# Router and API tests run in mock mode and catch basic regressions quickly.
!AIC_MODE=mock pytest -q

In [ ]:
import subprocess
import time
import requests

server_log = open("/kaggle/working/aic_demo_server.log", "w", encoding="utf-8")
server = subprocess.Popen(
    [sys.executable, "run.py"],
    cwd=PROJECT_ROOT,
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

for _ in range(120):
    try:
        health = requests.get("http://127.0.0.1:8000/api/health", timeout=2)
        if health.ok:
            break
    except Exception:
        pass
    time.sleep(1)
else:
    server.terminate()
    server_log.close()
    raise RuntimeError(Path("/kaggle/working/aic_demo_server.log").read_text(errors="ignore")[-5000:])

print("SERVER PID:", server.pid)
print(health.json())

In [ ]:
payload = {
    "query": "người đánh trống Yamaha ngoài trời",
    "profile": "auto",
    "top_k": 10,
    "ocr": "auto",
    "asr": "auto",
    "api_planner": "off",
}
response = requests.post(
    "http://127.0.0.1:8000/api/search",
    json=payload,
    timeout=120,
)
response.raise_for_status()
result = response.json()
print("LATENCY:", result["latency_ms"])
print("ROUTE OCR:", result["route"]["ocr"])
print("TOP HIT:", result["hits"][0] if result["hits"] else None)

In [ ]:
benchmark_queries = [
    "người mặc áo đỏ đứng cạnh ô tô",
    "người đánh trống Yamaha ngoài trời",
    "xe máy có biển số bắt đầu bằng 59",
    "người đàn ông nói về thời tiết",
]

for profile in ("fast", "auto"):
    benchmark = requests.post(
        "http://127.0.0.1:8000/api/benchmark",
        json={
            "queries": benchmark_queries,
            "profile": profile,
            "repeats": 1,
            "top_k": 10,
        },
        timeout=600,
    )
    benchmark.raise_for_status()
    print(profile.upper(), benchmark.json()["summary"])

## Optional public demo

FastAPI đang chạy nội bộ tại `127.0.0.1:8000`. Để lấy một share link nhanh trên Kaggle, dừng FastAPI hoặc mở cổng khác rồi chạy:

```bash
python scripts/gradio_demo.py
```

UI GitHub Pages nằm trong thư mục `docs/` và có thể trỏ tới một backend FastAPI được deploy công khai.